In [ ]:
# helper function to determine the sign of a NUMBER
pos_or_neg = lambda number: -1 if number < 0 else 1

# Helper function to determine how many bytes does it take to store this number
def size_in_bytes(number):
    import math

    if number == 0:
        return 1
    bits_needed = math.ceil(math.log2(abs(number) + 1))
    return math.ceil(bits_needed / 8)

def add(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(a + b)
    evm.pc += 1
    evm.gas_dec(3)

def mul(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(a * b)
    evm.pc += 1
    evm.gas_dec(5)

def sub(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(a - b)
    evm.pc += 1
    evm.gas_dec(3)

def div(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(0 if b == 0 else a // b)
    evm.pc += 1
    evm.gas_dec(5)

def sdiv(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    sign = pos_or_neg(a * b)
    evm.stack.push(0 if b == 0 else sign * (abs(a) // abs(b)))
    evm.pc += 1
    evm.gas_dec(5)

def mod(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(0 if b == 0 else a % b)
    evm.pc += 1
    evm.gas_dec(5)

def smod(evm):
    a, b = evm.stack.pop(), evm.stack.pop()
    sign = pos_or_neg(a)
    evm.stack.push(0 if b == 0 else sign * (abs(a) % abs(b)))
    evm.pc += 1
    evm.gas_dec(5)

def addmod(evm):
    a, b, N = evm.stack.pop(), evm.stack.pop(), evm.stack.pop()
    evm.stack.push(0 if N == 0 else (a + b) % N)
    evm.pc += 1
    evm.gas_dec(8)

def mulmod(evm):
    a, b, N = evm.stack.pop(), evm.stack.pop(), evm.stack.pop()
    evm.stack.push(0 if N == 0 else (a * b) % N)
    evm.pc += 1
    evm.gas_dec(8)

def exp(evm):
    a, exponent = evm.stack.pop(), evm.stack.pop()
    evm.stack.push(a ** exponent)
    evm.pc += 1
    evm.gas_dec(10 + (50 * size_in_bytes(exponent)))

def signextend(evm):
    b, x = evm.stack.pop(), evm.stack.pop()  # ✅ pops both b and x
    if b <= 31:                                # ✅ validates byte range
        testbit = b * 8 + 7                   # ✅ finds sign bit position
        sign_bit = 1 << testbit               # ✅ creates sign bit mask
        if x & sign_bit:
            result = x | (2**256 - sign_bit)  # ✅ extends with 1s (negative)
        else:
            result = x & (sign_bit - 1)       # ✅ extends with 0s (positive)
    else:
        result = x                             # ✅ already full size
    evm.stack.push(result)                     # ✅ pushes extended value
    evm.pc += 1
    evm.gas_dec(5)
